# 🤝 Multi-Agent Systems

## What are Multi-Agent Systems?
Multiple specialized AI agents working **together** to solve complex problems.

## Why Multiple Agents?
Single agent limitations:
- Context window limits
- Jack of all trades, master of none
- Hard to parallelize

Multiple agents:
- Each specializes in one domain
- Can work in parallel
- Divide and conquer

## Architecture Patterns
| Pattern | Description |
|---------|-------------|
| Supervisor | One agent delegates to specialists |
| Swarm | Agents hand off to each other |
| Parallel | Multiple agents run simultaneously |
| Hierarchical | Nested supervisor structure |


In [ ]:
# ── Multi-Agent Research System ────────────────────────────────────────
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage

# Shared state
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    topic: str
    research_notes: list[str]
    final_report: str

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Agent 1: Researcher
def researcher_agent(state: ResearchState) -> dict:
    prompt = f'Research the topic: {state["topic"]}. Provide 3 key facts.'
    response = llm.invoke([
        SystemMessage(content='You are an expert researcher. Be concise and factual.'),
        HumanMessage(content=prompt)
    ])
    return {
        'research_notes': [response.content],
        'messages': [AIMessage(content=f'[Researcher]: {response.content}')]
    }

# Agent 2: Analyst
def analyst_agent(state: ResearchState) -> dict:
    notes = '\n'.join(state['research_notes'])
    response = llm.invoke([
        SystemMessage(content='You are an analyst. Synthesize research into insights.'),
        HumanMessage(content=f'Analyze these notes and add insights:\n{notes}')
    ])
    return {
        'research_notes': state['research_notes'] + [response.content],
        'messages': [AIMessage(content=f'[Analyst]: {response.content}')]
    }

# Agent 3: Writer
def writer_agent(state: ResearchState) -> dict:
    notes = '\n'.join(state['research_notes'])
    response = llm.invoke([
        SystemMessage(content='You are a technical writer. Write a clear summary report.'),
        HumanMessage(content=f'Write a 2-paragraph report from:\n{notes}')
    ])
    return {
        'final_report': response.content,
        'messages': [AIMessage(content=f'[Writer]: Report complete.')]
    }

# Build graph
graph = StateGraph(ResearchState)
graph.add_node('researcher', researcher_agent)
graph.add_node('analyst', analyst_agent)
graph.add_node('writer', writer_agent)
graph.set_entry_point('researcher')
graph.add_edge('researcher', 'analyst')
graph.add_edge('analyst', 'writer')
graph.add_edge('writer', END)
pipeline = graph.compile()

result = pipeline.invoke({
    'messages': [],
    'topic': 'Large Language Models',
    'research_notes': [],
    'final_report': ''
})

print('=== FINAL REPORT ===')
print(result['final_report'])

## 🎯 Interview Questions

1. **What are multi-agent systems and why use them?**
2. **Name 4 multi-agent architecture patterns.**
3. **How do agents communicate in LangGraph?**
4. **What is the tradeoff of more agents?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Multi-Agent Systems — Collaboration at Scale**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
